In [ ]:
import tkinter as tk
from tkinter import messagebox, ttk, font as tkFont
import csv
import os
from datetime import datetime
import spacy
import networkx as nx
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
import logging
import glob
from PIL import Image, ImageTk

# Set up logging
logging.basicConfig(level=logging.DEBUG, format="%(asctime)s - %(levelname)s - %(message)s")

# Load SpaCy model
nlp = spacy.load("en_core_web_sm")

# Sentiment classifier data
sentiment_data = [
    ("I love this topic!", "positive"),
    ("This is so interesting!", "positive"),
    ("Great explanation!", "positive"),
    ("I'm confused", "negative"),
    ("Don't get this.", "negative"),
    ("This is too hard.", "negative"),
    ("What is this?", "neutral"),
    ("Can you explain?", "neutral"),
    ("Tell me about AI.", "neutral"),
    ("What are courses?", "neutral"),
    ("Phishing definition.", "neutral"),
    ("List courses.", "neutral"),
    ("Help with phishing.", "negative"),
    ("Awesome job!", "positive"),
]
sentiment_texts, sentiment_labels = zip(*sentiment_data)
vectorizer = TfidfVectorizer()
X_sentiment = vectorizer.fit_transform(sentiment_texts)
sentiment_classifier = MultinomialNB()
sentiment_classifier.fit(X_sentiment, sentiment_labels)

# Load knowledge graph
def load_knowledge_graph(csv_file_path="Knowledge_Graph_Dataset.csv"):
    knowledge_graph = {
        "Cyber Security": {"description": "", "why": "", "questions": {}, "courses": {}},
        "Machine Learning": {"description": "", "why": "", "questions": {}, "courses": {}},
        "Data Science": {"description": "", "why": "", "questions": {}, "courses": {}},
        "Introduction to AI": {"description": "", "why": "", "questions": {}, "courses": {}},
        "Networking": {"description": "", "why": "", "questions": {}, "courses": {}},
    }
    G = nx.DiGraph()
    if not os.path.exists(csv_file_path):
        messagebox.showerror("Error", f"CSV file '{csv_file_path}' not found.")
        return knowledge_graph, G
    try:
        with open(csv_file_path, mode='r', encoding='utf-8') as file:
            reader = csv.DictReader(file)
            for row in reader:
                field = row['Field']
                subfield = row['Subfield']
                topic = row['Topic']
                description = row['Description']
                if field in knowledge_graph:
                    if subfield == "Overview":
                        if topic.lower() == "description":
                            knowledge_graph[field]["description"] = description or "No description available."
                        elif topic.lower() == "why":
                            knowledge_graph[field]["why"] = description or "No reason provided."
                    elif subfield == "Concepts":
                        knowledge_graph[field]["questions"][topic.lower()] = description or "No details available."
                        G.add_node(topic.lower(), field=field, type="concept", description=description or "")
                    elif subfield == "Courses":
                        knowledge_graph[field]["courses"][topic.lower()] = description or "No course details available."
                        G.add_node(topic.lower(), field=field, type="course", description=description or "")
                    elif subfield == "Relationships":
                        try:
                            source, relation, target = topic.split("-")
                            G.add_edge(source.lower(), target.lower(), relation=relation.lower(), weight=1.0)
                            G.add_edge(target.lower(), source.lower(), relation=f"reverse-{relation.lower()}", weight=0.5)
                        except ValueError:
                            logging.warning(f"Invalid relationship format in row: {row}")
    except Exception as e:
        messagebox.showerror("Error", f"Error reading CSV: {e}")
        logging.error(f"CSV parsing error: {e}")
    return knowledge_graph, G

# Save new question to CSV
def save_new_question(field, question, answer, csv_file_path="Knowledge_Graph_Dataset.csv"):
    try:
        with open(csv_file_path, mode='a', encoding='utf-8', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([field, "Concepts", question, answer])
        logging.debug(f"Saved question '{question}' in '{field}'")
        return True
    except Exception as e:
        logging.error(f"Error saving question: {e}")
        messagebox.showerror("Error", f"Failed to save question: {e}")
        return False

# Process query with NLP
def process_query(query):
    doc = nlp(query.lower())
    keywords = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
    if not keywords:
        keywords = [query.lower().strip().replace(" ", "")]
    logging.debug(f"Processed query '{query}' to keywords: {keywords}")
    return keywords

# Detect sentiment
def get_sentiment(query):
    vector = vectorizer.transform([query])
    sentiment = sentiment_classifier.predict(vector)[0]
    logging.debug(f"Sentiment for '{query}': {sentiment}")
    return sentiment

# Adjust response based on sentiment
def adjust_response(response, sentiment):
    if sentiment == "negative":
        return f"Sorry you're finding this tough. Here's help: {response}"
    elif sentiment == "positive":
        return f"Glad you're excited! {response}"
    return response

# Best First Search
def best_first_search(query_keywords, graph, knowledge_graph):
    from heapq import heappush, heappop
    visited = set()
    queue = []
    for node in sorted(graph.nodes):
        keyword_score = sum(1 for keyword in query_keywords if keyword in node.lower())
        field = graph.nodes[node].get("field", "")
        field_score = 1 if any(keyword in field.lower() for keyword in query_keywords) else 0
        edge_bonus = sum(graph[pred][node].get("weight", 0)
                        for pred in graph.predecessors(node)
                        if any(keyword in pred.lower() for keyword in query_keywords))
        total_score = keyword_score + field_score + edge_bonus
        heappush(queue, (-total_score, node, [node]))
    while queue:
        _, node, path = heappop(queue)
        if node in visited:
            continue
        visited.add(node)
        node_data = graph.nodes[node]
        field = node_data.get("field")
        node_type = node_data.get("type")
        if field and node_type:
            if node_type == "concept" and node.lower() in knowledge_graph[field]["questions"]:
                return knowledge_graph[field]["questions"][node.lower()], path
            elif node_type == "course" and node.lower() in knowledge_graph[field]["courses"]:
                return knowledge_graph[field]["courses"][node.lower()], path
        for neighbor in sorted(graph.neighbors(node)):
            if neighbor not in visited:
                keyword_score = sum(1 for keyword in query_keywords if keyword in neighbor.lower())
                field = graph.nodes[neighbor].get("field", "")
                field_score = 1 if any(keyword in field.lower() for keyword in query_keywords) else 0
                edge_bonus = sum(graph[pred][neighbor].get("weight", 0)
                                for pred in graph.predecessors(neighbor)
                                if any(keyword in pred.lower() for keyword in query_keywords))
                total_score = keyword_score + edge_bonus + field_score
                heappush(queue, (-total_score, neighbor, path + [neighbor]))
    return "No relevant topics found.", []

# Get response
def get_real_time_response(query, knowledge_graph, graph):
    sentiment = get_sentiment(query)
    keywords = process_query(query)
    query_field = query.lower().strip()
    if query_field in knowledge_graph:
        overview = knowledge_graph[query_field]["description"]
        if overview:
            return adjust_response(f"Topic: {query_field.capitalize()}\n\nAnswer: {overview}", sentiment), ""
    if "course" in keywords or query.lower().strip() == "courses":
        course_list = []
        for field, details in knowledge_graph.items():
            for course, description in details["courses"].items():
                course_list.append(f"Course: {course.capitalize()}\nDetails: {description}")
        if course_list:
            response = "\n\n".join(course_list)
            return adjust_response(response, sentiment), []
        return adjust_response("No courses found.", sentiment), []
    for field, details in knowledge_graph.items():
        for keyword, answer in details["questions"].items():
            if any(k.lower() == keyword.lower() for k in keywords) or (len(keywords) == 1 and keywords[0] in keyword.lower()):
                return adjust_response(f"Topic: {field}\n\nAnswer: {answer}", sentiment), []
        for keyword, answer in details["courses"].items():
            if any(k.lower() == keyword.lower() for k in keywords) or (len(keywords) == 1 and keywords[0] in keyword.lower()):
                return adjust_response(f"Course: {field}\n\nDetails: {answer}", sentiment), []
    answer, path = best_first_search(keywords, graph, knowledge_graph)
    return adjust_response(answer, sentiment), path

# Chatbot GUI
class RealTimeTutorBot:
    def __init__(self, root):
        self.root = root
        self.root.title("Intelligent Personal Tutor Bot by AR")
        self.root.geometry("1000x700")
        self.button_font = tkFont.Font(family="Arial", size=11, weight="bold")
        self.themes = {
            "Light": {
                "bg": "#e0f7fa", "fg": "black", "text_bg": "white", "text_fg": "black",
                "button_bg": "#8B0000", "button_fg": "white", "button_hover_bg": "#A52A2A",
                "button_active_bg": "#6B0000", "label_fg": "#333333", "entry_bg": "white",
                "entry_fg": "black",
            },
            "Dark": {
                "bg": "#2d2d2d", "fg": "white", "text_bg": "#3c3f41", "text_fg": "white",
                "button_bg": "#8B0000", "button_fg": "white", "button_hover_bg": "#A52A2A",
                "button_active_bg": "#6B0000", "label_fg": "#bbbbbb", "entry_bg": "#4b4b4b",
                "entry_fg": "white",
            },
            "Blue": {
                "bg": "#263238", "fg": "white", "text_bg": "#37474f", "text_fg": "white",
                "button_bg": "#8B0000", "button_fg": "white", "button_hover_bg": "#A52A2A",
                "button_active_bg": "#6B0000", "label_fg": "#90caf9", "entry_bg": "#455a64",
                "entry_fg": "white",
            }
        }
        self.knowledge_graph, self.graph = load_knowledge_graph()
        self.current_theme = "Light"
        self.root.configure(bg=self.themes[self.current_theme]["bg"])

        # Header Frame for title and logo
        self.header_frame = tk.Frame(self.root, bg=self.themes[self.current_theme]["bg"])
        self.header_frame.pack(pady=10)

        # Title Label
        self.title_label = tk.Label(self.header_frame, text="Intelligent Tutor Bot", font=("Helvetica", 18, "bold"),
                                    bg=self.themes[self.current_theme]["bg"],
                                    fg=self.themes[self.current_theme]["label_fg"])
        self.title_label.pack(side=tk.LEFT, padx=10)

        # Logo Image
        image_path = "logo.PNG"
        try:
            image = Image.open(image_path)
            image = image.resize((100, 50), Image.Resampling.LANCZOS)
            self.logo = ImageTk.PhotoImage(image)
            self.logo_label = tk.Label(self.header_frame, image=self.logo, bg=self.themes[self.current_theme]["bg"])
            self.logo_label.image = self.logo  # Prevent garbage collection
            self.logo_label.pack(side=tk.LEFT, padx=10)
        except Exception as e:
            logging.error(f"Failed to load image at {image_path}: {e}")
            self.logo_label = tk.Label(self.header_frame, text="[Logo]", font=("Arial", 12),
                                       bg=self.themes[self.current_theme]["bg"],
                                       fg=self.themes[self.current_theme]["label_fg"])
            self.logo_label.pack(side=tk.LEFT, padx=10)

        self.prompt_label = tk.Label(self.root, text="Ask about Cyber Security, Machine Learning, Data Science, AI, Networking, or courses:",
                                    font=("Arial", 14, "bold"), bg=self.themes[self.current_theme]["bg"],
                                    fg=self.themes[self.current_theme]["label_fg"])
        self.prompt_label.pack(pady=10)

        self.text_frame = tk.Frame(self.root, bg=self.themes[self.current_theme]["bg"])
        self.text_frame.pack(pady=10, fill=tk.BOTH, expand=True)
        self.text_display = tk.Text(self.text_frame, height=20, width=110, wrap=tk.WORD, font=("Arial", 11),
                                   bg=self.themes[self.current_theme]["text_bg"],
                                   fg=self.themes[self.current_theme]["text_fg"])
        self.text_display.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        self.scrollbar = tk.Scrollbar(self.text_frame, orient=tk.VERTICAL, command=self.text_display.yview,
                                      bg=self.themes[self.current_theme]["bg"])
        self.scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        self.text_display.config(yscrollcommand=self.scrollbar.set)

        self.button_frame = tk.Frame(self.root, bg=self.themes[self.current_theme]["bg"])
        self.button_frame.pack(pady=5, fill=tk.X)
        self.reset_btn = tk.Button(self.button_frame, text="Reset Session", command=self.reset_session,
                                  bg=self.themes[self.current_theme]["button_bg"],
                                  fg=self.themes[self.current_theme]["button_fg"],
                                  font=self.button_font, borderwidth=1, relief="solid")
        self.reset_btn.pack(side=tk.LEFT, padx=10)
        self.add_button_effects(self.reset_btn)
        self.new_session_btn = tk.Button(self.button_frame, text="New Session", command=self.new_session,
                                        bg=self.themes[self.current_theme]["button_bg"],
                                        fg=self.themes[self.current_theme]["button_fg"],
                                        font=self.button_font, borderwidth=1, relief="solid")
        self.new_session_btn.pack(side=tk.LEFT, padx=10)
        self.add_button_effects(self.new_session_btn)
        self.save_btn = tk.Button(self.button_frame, text="Save Session", command=self.save_session,
                                 bg=self.themes[self.current_theme]["button_bg"],
                                 fg=self.themes[self.current_theme]["button_fg"],
                                 font=self.button_font, borderwidth=1, relief="solid")
        self.save_btn.pack(side=tk.LEFT, padx=10)
        self.add_button_effects(self.save_btn)
        self.history_btn = tk.Button(self.button_frame, text="View History", command=self.view_history,
                                    bg=self.themes[self.current_theme]["button_bg"],
                                    fg=self.themes[self.current_theme]["button_fg"],
                                    font=self.button_font, borderwidth=1, relief="solid")
        self.history_btn.pack(side=tk.LEFT, padx=10)
        self.add_button_effects(self.history_btn)
        self.teach_btn = tk.Button(self.button_frame, text="Teach Bot", command=self.teach_bot,
                                   bg=self.themes[self.current_theme]["button_bg"],
                                   fg=self.themes[self.current_theme]["button_fg"],
                                   font=self.button_font, borderwidth=1, relief="solid")
        self.teach_btn.pack(side=tk.LEFT, padx=10)
        self.add_button_effects(self.teach_btn)
        self.theme_var = tk.StringVar(value=self.current_theme)
        self.theme_combobox = ttk.Combobox(self.button_frame, textvariable=self.theme_var, values=list(self.themes.keys()),
                                           state="readonly", width=10)
        self.theme_combobox.pack(side=tk.LEFT, padx=10)
        self.theme_combobox.bind("<<ComboboxSelected>>", self.change_theme)

        self.input_frame = tk.Frame(self.root, bg=self.themes[self.current_theme]["bg"])
        self.input_frame.pack(pady=10, fill=tk.X)
        self.user_input = tk.Entry(self.input_frame, font=("Arial", 12), width=90,
                                   bg=self.themes[self.current_theme]["entry_bg"],
                                   fg=self.themes[self.current_theme]["entry_fg"])
        self.user_input.pack(side=tk.LEFT, padx=20)
        self.user_input.bind("<KeyRelease>", self.show_suggestions)
        self.user_input.bind("<FocusOut>", self.hide_suggestions)
        self.user_input.bind("<Return>", self.select_suggestion)
        self.send_btn = tk.Button(self.input_frame, text="Send", command=self.process_input,
                                 bg=self.themes[self.current_theme]["button_bg"],
                                 fg=self.themes[self.current_theme]["button_fg"],
                                 font=self.button_font, borderwidth=1, relief="solid")
        self.send_btn.pack(side=tk.LEFT)
        self.add_button_effects(self.send_btn)

        self.suggestion_list = tk.Listbox(self.root, font=("Arial", 12), height=5,
                                         bg=self.themes[self.current_theme]["text_bg"],
                                         fg=self.themes[self.current_theme]["text_fg"],
                                         selectbackground=self.themes[self.current_theme]["button_bg"],
                                         selectforeground="white")
        self.suggestion_list.bind("<<ListboxSelect>>", self.select_suggestion)
        self.suggestion_list.bind("<Double-1>", self.select_suggestion)
        self.suggestion_list.place_forget()

    def add_button_effects(self, button):
        button.bind("<Enter>", lambda e: button.configure(bg=self.themes[self.current_theme]["button_hover_bg"]))
        button.bind("<Leave>", lambda e: button.configure(bg=self.themes[self.current_theme]["button_bg"]))
        button.bind("<Button-1>", lambda e: self.animate_button_click(button))
        button.bind("<ButtonPress-1>", lambda e: button.configure(bg=self.themes[self.current_theme]["button_active_bg"]))
        button.bind("<ButtonRelease-1>", lambda e: button.configure(bg=self.themes[self.current_theme]["button_hover_bg"] if button.winfo_containing(button.winfo_pointerx(), button.winfo_pointery()) == button else self.themes[self.current_theme]["button_bg"]))

    def animate_button_click(self, button):
        original_width = button.winfo_width()
        original_height = button.winfo_height()
        button.configure(width=int(original_width * 1.1), height=int(original_height * 1.1))
        self.root.after(100, lambda: button.configure(width=original_width, height=original_height))

    def show_suggestions(self, event=None):
        query = self.user_input.get().lower().strip()
        if not query:
            self.hide_suggestions()
            return
        suggestions = set()
        for field, details in self.knowledge_graph.items():
            if query in field.lower():
                suggestions.add(field)
            for concept in details["questions"]:
                if query in concept.lower():
                    suggestions.add(concept)
            for course in details["courses"]:
                if query in course.lower():
                    suggestions.add(course)
        suggestions = sorted(list(suggestions))[:5]
        self.suggestion_list.delete(0, tk.END)
        for suggestion in suggestions:
            self.suggestion_list.insert(tk.END, suggestion)
        if suggestions:
            x = self.user_input.winfo_rootx()
            y = self.user_input.winfo_rooty() + self.user_input.winfo_height()
            self.suggestion_list.place(x=x, y=y, width=self.suggestion_list.winfo_width())
        else:
            self.hide_suggestions()

    def hide_suggestions(self):
        self.suggestion_list.place_forget()

    def select_suggestion(self, event=None):
        if self.suggestion_list.winfo_ismapped() and self.suggestion_list.curselection():
            selected = self.suggestion_list.get(self.suggestion_list.curselection()[0])
            self.user_input.delete(0, tk.END)
            self.user_input.insert(0, selected)
            self.hide_suggestions()
        elif event and event.keysym == "Return" and self.user_input.get().strip():
            self.process_input()
        return "break"

    def teach_bot(self):
        teach_window = tk.Toplevel(self.root)
        teach_window.title("Teach Bot")
        teach_window.geometry("400x400")
        teach_window.configure(bg=self.themes[self.current_theme]["bg"])
        tk.Label(teach_window, text="Teach a New Question", font=("Arial", 14),
                 bg=self.themes[self.current_theme]["bg"],
                 fg=self.themes[self.current_theme]["label_fg"]).pack(pady=10)
        tk.Label(teach_window, text="Field:", font=("Arial", 12),
                 bg=self.themes[self.current_theme]["bg"],
                 fg=self.themes[self.current_theme]["label_fg"]).pack()
        field_var = tk.StringVar()
        field_menu = ttk.Combobox(teach_window, textvariable=field_var, values=list(self.knowledge_graph.keys()),
                                   state="readonly", width=30)
        field_menu.pack(pady=5)
        tk.Label(teach_window, text="Question:", font=("Arial", 12),
                 bg=self.themes[self.current_theme]["bg"],
                 fg=self.themes[self.current_theme]["label_fg"]).pack()
        question_entry = tk.Entry(teach_window, font=("Arial", 12), width=40,
                                  bg=self.themes[self.current_theme]["entry_bg"],
                                  fg=self.themes[self.current_theme]["entry_fg"])
        question_entry.pack(pady=5)
        tk.Label(teach_window, text="Answer:", font=("Arial", 12),
                 bg=self.themes[self.current_theme]["bg"],
                 fg=self.themes[self.current_theme]["label_fg"]).pack()
        answer_text = tk.Text(teach_window, font=("Arial", 12), height=5, width=40,
                             bg=self.themes[self.current_theme]["entry_bg"],
                             fg=self.themes[self.current_theme]["entry_fg"])
        answer_text.pack(pady=5)
        def submit_question():
            field = field_var.get()
            question = question_entry.get().strip()
            answer = answer_text.get("1.0", tk.END).strip()
            if not field:
                messagebox.showerror("Error", "Select a field.")
                return
            if not question:
                messagebox.showerror("Error", "Enter a question.")
                return
            if not answer:
                messagebox.showerror("Error", "Enter an answer.")
                return
            if save_new_question(field, question, answer):
                self.knowledge_graph[field]["questions"][question.lower()] = answer
                self.graph.add_node(question.lower(), field=field, type="concept", description=answer)
                messagebox.showinfo("Success", f"Question '{question}' added to {field}.")
                teach_window.destroy()
                self.show_suggestions()
        submit_btn = tk.Button(teach_window, text="Submit", command=submit_question,
                              bg=self.themes[self.current_theme]["button_bg"],
                              fg=self.themes[self.current_theme]["button_fg"],
                              font=self.button_font, borderwidth=1, relief="solid")
        submit_btn.pack(pady=10)
        self.add_button_effects(submit_btn)

    def change_theme(self, event=None):
        self.current_theme = self.theme_var.get()
        theme = self.themes[self.current_theme]
        self.root.configure(bg=theme["bg"])
        self.header_frame.configure(bg=theme["bg"])
        self.title_label.configure(bg=theme["bg"], fg=theme["label_fg"])
        self.logo_label.configure(bg=theme["bg"])
        self.prompt_label.configure(bg=theme["bg"], fg=theme["label_fg"])
        self.text_frame.configure(bg=theme["bg"])
        self.text_display.configure(bg=theme["text_bg"], fg=theme["text_fg"])
        self.scrollbar.configure(bg=theme["bg"])
        self.button_frame.configure(bg=theme["bg"])
        self.input_frame.configure(bg=theme["bg"])
        self.user_input.configure(bg=theme["entry_bg"], fg=theme["entry_fg"])
        self.suggestion_list.configure(bg=theme["text_bg"], fg=theme["text_fg"],
                                    selectbackground=theme["button_bg"], selectforeground="white")
        for btn in [self.reset_btn, self.new_session_btn, self.save_btn, self.history_btn, self.teach_btn, self.send_btn]:
            btn.configure(bg=theme["button_bg"], fg=theme["button_fg"],
                         activebackground=theme["button_active_bg"], activeforeground=theme["button_fg"])
        for window in self.root.winfo_children():
            if isinstance(window, tk.Toplevel):
                window.configure(bg=theme["bg"])
                for widget in window.winfo_children():
                    if isinstance(widget, tk.Label):
                        widget.configure(bg=theme["bg"], fg=theme["label_fg"])
                    elif isinstance(widget, tk.Frame):
                        widget.configure(bg=theme["bg"])
                    elif isinstance(widget, tk.Text):
                        widget.configure(bg=theme["text_bg"], fg=theme["text_fg"])
                    elif isinstance(widget, tk.Listbox):
                        widget.configure(bg=theme["text_bg"], fg=theme["text_fg"])
                    elif isinstance(widget, tk.Entry):
                        widget.configure(bg=theme["entry_bg"], fg=theme["entry_fg"])
                    elif isinstance(widget, tk.Button):
                        widget.configure(bg=theme["button_bg"], fg=theme["button_fg"],
                                        activebackground=theme["button_active_bg"],
                                        activeforeground=theme["button_fg"])

    def process_input(self):
        query = self.user_input.get()
        if query:
            self.text_display.insert(tk.END, f"You: {query}\n")
            response, path = get_real_time_response(query, self.knowledge_graph, self.graph)
            self.text_display.insert(tk.END, f"Bot: {response}\n")
            if path:
                self.text_display.insert(tk.END, f"(Path: {' -> '.join(path)})\n")
            self.text_display.insert(tk.END, "\n")
            self.text_display.see(tk.END)
            self.user_input.delete(0, tk.END)
            self.hide_suggestions()

    def reset_session(self):
        self.text_display.delete(1.0, tk.END)
        self.text_display.insert(tk.END, "Session reset.\n\n")

    def new_session(self):
        self.save_session()
        self.text_display.delete(1.0, tk.END)
        self.text_display.insert(tk.END, "New session started.\n\n")

    def save_session(self):
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"chat_session_{timestamp}.txt"
            with open(filename, "w", encoding="utf-8") as file:
                file.write(self.text_display.get("1.0", tk.END))
            messagebox.showinfo("Success", f"Session saved as {filename}")
        except Exception as e:
            messagebox.showerror("Error", f"Failed to save: {e}")

    def view_history(self):
        history_window = tk.Toplevel(self.root)
        history_window.title("History")
        history_window.geometry("600x400")
        history_window.configure(bg=self.themes[self.current_theme]["bg"])
        tk.Label(history_window, text="Select Session", font=("Arial", 14, "bold"),
                 bg=self.themes[self.current_theme]["bg"],
                 fg=self.themes[self.current_theme]["label_fg"]).pack(pady=10)
        list_frame = tk.Frame(history_window, bg=self.themes[self.current_theme]["bg"])
        list_frame.pack(pady=10, fill=tk.BOTH, expand=True)
        scrollbar = tk.Scrollbar(list_frame, orient=tk.VERTICAL,
                                    bg=self.themes[self.current_theme]["bg"])
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        session_listbox = tk.Listbox(list_frame, font=("Arial", 12), width=50, height=15,
                            yscrollcommand=scrollbar.set,
                            bg=self.themes[self.current_theme]["text_bg"],
                            fg=self.themes[self.current_theme]["text_fg"])
        session_listbox.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scrollbar.config(command=session_listbox.yview)
        session_files = glob.glob("chat_session_*.txt")
        if not session_files:
            session_listbox.insert(tk.END, "No sessions found.")
        else:
            for file in sorted(session_files, key=os.path.getmtime, reverse=True):
                session_listbox.insert(tk.END, file)
        def load_session(event):
            selected = session_listbox.get(session_listbox.curselection())
            if selected and selected != "No sessions found.":
                try:
                    with open(selected, "r", encoding="utf-8") as file:
                        content = file.read()
                    session_window = tk.Toplevel(history_window)
                    session_window.title(f"Session: {selected}")
                    session_window.geometry("600x400")
                    session_window.configure(bg=self.themes[self.current_theme]["bg"])
                    text_display = tk.Text(session_window, wrap=tk.WORD, font=("Arial", 11),
                                     bg=self.themes[self.current_theme]["text_bg"],
                                     fg=self.themes[self.current_theme]["text_fg"])
                    text_display.pack(pady=10, padx=10, fill=tk.BOTH, expand=True)
                    text_display.insert(tk.END, content)
                    text_display.config(state=tk.DISABLED)
                except Exception as e:
                    messagebox.showerror("Error", f"Failed to load: {e}")
        session_listbox.bind("<<ListboxSelect>>", load_session)

# Launch
if __name__ == "__main__":
    root = tk.Tk()
    app = RealTimeTutorBot(root)
    root.mainloop()

2025-06-12 03:53:54,506 - DEBUG - STREAM b'IHDR' 16 13
2025-06-12 03:53:54,509 - DEBUG - STREAM b'sRGB' 41 1
2025-06-12 03:53:54,510 - DEBUG - STREAM b'gAMA' 54 4
2025-06-12 03:53:54,512 - DEBUG - STREAM b'pHYs' 70 9
2025-06-12 03:53:54,517 - DEBUG - STREAM b'IDAT' 91 56883
